# 🛰️ Lab 8: Deep Learning for Satellite Image Analysis
## Advanced Spatial Analysis — Southern Illinois University

---

| | |
|---|---|
| **Course** | Advanced Spatial Analysis |
| **Lab** | Lab 8 — Deep Learning for Land Use Classification |
| **Platform** | Google Colab (no installation needed) |
| **Dataset** | EuroSAT — Sentinel-2 Satellite Imagery |
| **Estimated Time** | 3 – 4 hours |
| **Skill Level** | Beginner-friendly (no prior coding required) |

---

### 🎯 Learning Objectives

By the end of this lab you will be able to:

1. Navigate Google Colab and run Python code in a notebook environment
2. Understand and use key Python packages for image analysis (`numpy`, `matplotlib`, `tensorflow`)
3. Explain what a Convolutional Neural Network (CNN) is and how it classifies satellite images
4. Train a CNN to classify Sentinel-2 land use/land cover (LULC) classes from the EuroSAT dataset
5. Understand how a U-Net differs from a CNN and why it is used for pixel-level land cover mapping
6. Apply a U-Net model to produce a land cover segmentation map from Sentinel-2 imagery

---

### 📋 What You Will Submit

- This completed notebook (File → Download → Download .ipynb) with all cells run
- Screenshots of your training accuracy curve and final prediction maps
- Written answers to the **Reflection Questions** at the end of each section

---

> **How to use this notebook:** Read every text block carefully before running the code below it.
> To run a code cell, click on it and press **Shift + Enter** (or click the ▶ play button on the left).
> Never skip a cell — each one builds on the previous.


---
# Part 0: Getting Started with Google Colab

---

## 0.1 What is Google Colab?

Google Colab (short for Colaboratory) is a free, cloud-based environment where you can write and run Python code directly in your browser — no software installation required. It gives you access to a real computer (including a GPU, a specialized processor for deep learning) hosted by Google.

A Colab notebook is made up of two types of **cells**:

- **Text cells** (like this one) — contain explanations, instructions, and questions. Written in Markdown.
- **Code cells** — contain Python code. You run them by pressing **Shift + Enter**.

### ⚡ Enable GPU (Important — do this first!)

Deep learning models train much faster on a GPU. To enable it:

1. Click **Runtime** in the top menu
2. Click **Change runtime type**
3. Under **Hardware accelerator**, select **T4 GPU**
4. Click **Save**

You should see a small green indicator at the top right of the page confirming GPU is connected.


In [1]:
# ── Cell 0.1 ── Check that our environment is ready
# Run this cell first. It will tell us what hardware we are using.

import platform
print("Python version:", platform.python_version())

# Check for GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print("\n✅ GPU detected! Your training will be fast.")
    print(result.stdout.split('\n')[8])  # Print GPU model line
else:
    print("\n⚠️  No GPU detected. Go to Runtime > Change runtime type > T4 GPU")


Python version: 3.8.20

✅ GPU detected! Your training will be fast.
|   0  NVIDIA GeForce GTX 1660      WDDM  |   00000000:01:00.0  On |                  N/A |


---
## 0.2 Python Basics — Just Enough to Read This Lab

You do not need to write Python from scratch in this lab — all code is provided. But you do need to **read** the code to understand what is happening. This section covers only the concepts you will encounter.

### Variables and printing


In [2]:
# ── Cell 0.2a ── Variables and printing
# A variable stores a value. The = sign assigns a value to a name.

course_name = "Advanced Spatial Analysis"
lab_number = 8
is_deep_learning = True

# print() displays output
print("Course:", course_name)
print("Lab number:", lab_number)
print("Is this deep learning?", is_deep_learning)


Course: Advanced Spatial Analysis
Lab number: 8
Is this deep learning? True


### Lists and loops

A **list** holds multiple items. A **loop** repeats an action for each item.


In [ ]:
# ── Cell 0.2b ── Lists and loops

# This is a list of EuroSAT land cover class names
land_cover_classes = [
    "Annual Crop", "Forest", "Herbaceous Vegetation",
    "Highway", "Industrial", "Pasture",
    "Permanent Crop", "Residential", "River", "Sea/Lake"
]

print(f"There are {len(land_cover_classes)} land cover classes in EuroSAT:\n")

# A for loop goes through each item in the list
for i, class_name in enumerate(land_cover_classes):
    print(f"  Class {i}: {class_name}")


There are 10 land cover classes in EuroSAT:

  Class 0: Annual Crop
  Class 1: Forest
  Class 2: Herbaceous Vegetation
  Class 3: Highway
  Class 4: Industrial
  Class 5: Pasture
  Class 6: Permanent Crop
  Class 7: Residential
  Class 8: River
  Class 9: Sea/Lake


### Functions

A **function** is a reusable block of code. You call it by name and pass it inputs.


In [4]:
# ── Cell 0.2c ── Functions

def describe_image(height, width, channels):
    """Describes the shape of a satellite image array."""
    total_pixels = height * width
    print(f"Image size: {height} x {width} pixels")
    print(f"Number of bands/channels: {channels}")
    print(f"Total pixels: {total_pixels:,}")

# Call the function with EuroSAT image dimensions
describe_image(64, 64, 13)   # EuroSAT patches are 64x64 with 13 Sentinel-2 bands


Image size: 64 x 64 pixels
Number of bands/channels: 13
Total pixels: 4,096


---
## 0.3 Key Python Packages

This lab uses four main packages. You don't need to memorize them — just know what each one does when you see it in the code.

| Package | What it does | How you'll see it |
|---|---|---|
| `numpy` | Fast math on large arrays of numbers | `import numpy as np` |
| `matplotlib` | Plotting and displaying images | `import matplotlib.pyplot as plt` |
| `tensorflow` / `keras` | Building and training deep learning models | `import tensorflow as tf` |
| `sklearn` | Evaluation metrics (accuracy, confusion matrix) | `from sklearn.metrics import ...` |

Run the cell below to import everything needed for the whole lab.


In [5]:
# ── Cell 0.3 ── Install and import all packages
# Most are pre-installed in Colab. We just need to import them.

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os, random, warnings
warnings.filterwarnings('ignore')

# TensorFlow / Keras — our deep learning framework
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Scikit-learn — for evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Set seeds so results are reproducible
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

print("✅ All packages imported successfully!")
print(f"   TensorFlow version: {tf.__version__}")
print(f"   NumPy version: {np.__version__}")


ModuleNotFoundError: No module named 'tensorflow'

---
## 0.4 How Images Are Stored as Numbers

Before we do any deep learning, it's important to understand how a computer sees a satellite image.

A satellite image is stored as a **3D array of numbers** with shape:
`[height, width, number_of_bands]`

- Each **pixel** is one location on the ground
- Each **band** records how much light was reflected at a particular wavelength
- Sentinel-2 has **13 spectral bands**, including visible (RGB), near-infrared, and shortwave infrared

Let's load one EuroSAT image and examine it.


In [ ]:
# ── Cell 0.4 ── Download EuroSAT dataset and explore image structure

# Download the RGB version of EuroSAT (easier to visualize, 3 bands)
# This is the standard dataset used in the gicait DL tutorial (S3)
import tensorflow_datasets as tfds

print("Downloading EuroSAT dataset... (this may take 2-3 minutes)")
dataset, info = tfds.load('eurosat/rgb', with_info=True, as_supervised=True)
print("\n✅ Download complete!")
print("\nDataset info:")
print(f"  Image shape: {info.features['image'].shape}")
print(f"  Number of classes: {info.features['label'].num_classes}")
print(f"  Class names: {info.features['label'].names}")


In [ ]:
# ── Cell 0.4b ── Visualize sample images from each class

CLASS_NAMES = info.features['label'].names
NUM_CLASSES = len(CLASS_NAMES)

# Collect one sample per class for display
samples = {name: None for name in CLASS_NAMES}
for image, label in dataset['train']:
    name = CLASS_NAMES[label.numpy()]
    if samples[name] is None:
        samples[name] = image.numpy()
    if all(v is not None for v in samples.values()):
        break

# Plot one image per class
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle("EuroSAT Dataset — One Sample per Land Cover Class\n(Sentinel-2 RGB Imagery)",
             fontsize=14, fontweight='bold', y=1.01)

for ax, (name, img) in zip(axes.flatten(), samples.items()):
    ax.imshow(img)
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('eurosat_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n💾 Figure saved as eurosat_samples.png")


In [ ]:
# ── Cell 0.4c ── Inspect the pixel values of one image

sample_image = list(samples.values())[1]  # Forest class
print("Image array shape:", sample_image.shape)
print("  → (height, width, channels) =", sample_image.shape)
print("\nPixel value range:", sample_image.min(), "to", sample_image.max())
print("Data type:", sample_image.dtype)

# Show the three RGB channels separately
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
channel_names = ['Red (Band 4)', 'Green (Band 3)', 'Blue (Band 2)']
cmaps = ['Reds', 'Greens', 'Blues']

axes[0].imshow(sample_image)
axes[0].set_title('RGB Composite', fontweight='bold')
axes[0].axis('off')

for i in range(3):
    im = axes[i+1].imshow(sample_image[:,:,i], cmap=cmaps[i])
    axes[i+1].set_title(channel_names[i], fontweight='bold')
    axes[i+1].axis('off')
    plt.colorbar(im, ax=axes[i+1], fraction=0.046)

plt.suptitle("Forest Image — RGB Composite and Individual Bands", fontweight='bold')
plt.tight_layout()
plt.show()


### 📝 Reflection Questions — Part 0

Answer these questions in the text cell below. Double-click the cell to edit it.


**Q0.1:** Look at the EuroSAT sample images above. Choose two land cover classes that look most similar to each other visually. Why might a computer have trouble telling them apart?

*Your answer here:*

---

**Q0.2:** A Sentinel-2 image patch in EuroSAT is 64 × 64 pixels and covers approximately 6,400 m² on the ground. What is the spatial resolution (meters per pixel) of these patches?

*Your answer here:*

---

**Q0.3:** The pixel values in the image range from 0 to 255. What does a pixel value of 0 mean? What does 255 mean?

*Your answer here:*


---
# Part 1: Convolutional Neural Networks (CNN) for Land Use Classification

---

## 1.1 What is a Convolutional Neural Network?

A **Convolutional Neural Network (CNN)** is a type of deep learning model designed specifically to work with images. Unlike a regular computer program that follows rules you write, a CNN **learns the rules automatically** by looking at thousands of labeled examples.

### How a CNN processes an image:

```
Input Image (64×64×3)
       ↓
[Convolutional Layer] — Detects simple features: edges, corners, color gradients
       ↓
[Pooling Layer] — Shrinks the image, keeps the important features
       ↓
[Convolutional Layer] — Detects complex features: textures, patterns
       ↓
[Pooling Layer]
       ↓
[Flatten] — Converts the 2D feature map into a 1D list of numbers
       ↓
[Dense Layer] — Combines all features to make a decision
       ↓
Output: Probability for each class → picks the highest one
```

**Key terms:**
- **Convolution:** Sliding a small filter (e.g. 3×3) across the image to detect features
- **Filter/Kernel:** A small matrix of weights the model learns during training
- **Pooling:** Reducing image size by taking the maximum value in each region (MaxPooling)
- **ReLU:** An activation function that introduces non-linearity (sets negative values to 0)
- **Softmax:** The final activation that converts raw scores into class probabilities (must sum to 1)

### Training process:
1. Show the CNN a labeled image (e.g. "this is Forest")
2. The CNN makes a prediction (e.g. "I think it's Residential — 72% confidence")
3. Calculate the error (loss) between the prediction and the truth
4. **Backpropagation:** Adjust the filter weights slightly to reduce the error
5. Repeat millions of times across thousands of images


## 1.2 Prepare the EuroSAT Data for Training


In [ ]:
# ── Cell 1.2 ── Load and preprocess EuroSAT for CNN training

IMG_SIZE = 64
BATCH_SIZE = 32

def preprocess(image, label):
    """Normalize pixel values from 0-255 to 0-1 range."""
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

# Split into train / validation / test
# EuroSAT has 21,600 training + 5,400 test images
train_ds = dataset['train'].map(preprocess)
test_ds  = dataset['test'].map(preprocess)

# Further split training into train + validation (80/20)
train_size = int(0.8 * 21600)
val_size   = 21600 - train_size

train_ds_final = train_ds.take(train_size).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds_final   = train_ds.skip(train_size).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds_final  = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Dataset split:")
print(f"  Training samples:   {train_size:,}")
print(f"  Validation samples: {val_size:,}")
print(f"  Test samples:       5,400")
print(f"\nBatch size: {BATCH_SIZE}")
print(f"Training batches per epoch: {train_size // BATCH_SIZE}")


## 1.3 Build the CNN Architecture

Now we define the structure of our CNN. Each layer is added one by one. Read the comments carefully — they explain what each layer does.


In [ ]:
# ── Cell 1.3 ── Define the CNN model

def build_cnn(input_shape=(64, 64, 3), num_classes=10):
    """
    A simple CNN for EuroSAT land use classification.
    Architecture inspired by the gicait DL-for-satellite-image-analysis tutorial (S3).
    """
    model = models.Sequential([

        # ── Block 1: First convolution ──────────────────────────────
        # 32 filters, each 3x3 pixels. Detects simple edges and color patterns.
        layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                      input_shape=input_shape, name='conv1_1'),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', name='conv1_2'),
        # MaxPooling reduces the 64x64 image to 32x32 — keeps the strongest features
        layers.MaxPooling2D((2, 2), name='pool1'),
        # Dropout randomly turns off 25% of neurons during training — prevents overfitting
        layers.Dropout(0.25, name='drop1'),

        # ── Block 2: Deeper features ─────────────────────────────────
        # 64 filters — now detecting more complex patterns like texture
        layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv2_1'),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv2_2'),
        # 32x32 → 16x16
        layers.MaxPooling2D((2, 2), name='pool2'),
        layers.Dropout(0.25, name='drop2'),

        # ── Block 3: Even deeper features ────────────────────────────
        layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='conv3_1'),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='conv3_2'),
        # 16x16 → 8x8
        layers.MaxPooling2D((2, 2), name='pool3'),
        layers.Dropout(0.4, name='drop3'),

        # ── Classification head ───────────────────────────────────────
        # Flatten converts the 3D feature map (8x8x128) into a 1D vector (8192 values)
        layers.Flatten(name='flatten'),
        # Dense layer: fully connected — every neuron connects to every feature
        layers.Dense(256, activation='relu', name='dense1'),
        layers.Dropout(0.5, name='drop4'),
        # Output layer: one neuron per class, softmax gives probabilities
        layers.Dense(num_classes, activation='softmax', name='output')
    ])
    return model

cnn_model = build_cnn()
cnn_model.summary()


In [ ]:
# ── Cell 1.3b ── Visualize the model architecture as a diagram

tf.keras.utils.plot_model(
    cnn_model,
    to_file='cnn_architecture.png',
    show_shapes=True,
    show_layer_names=True,
    rankdir='TB',
    dpi=80
)

from IPython.display import Image
Image('cnn_architecture.png')


## 1.4 Compile and Train the CNN

**Compiling** means telling the model:
- **Optimizer:** How to update the weights (we use `Adam` — a popular, adaptive optimizer)
- **Loss function:** How to measure the error (`categorical_crossentropy` — standard for multi-class classification)
- **Metric:** What to report during training (`accuracy` — % of correct predictions)

**Training** feeds the data through the model repeatedly. Each full pass through the training data is called an **epoch**.


In [ ]:
# ── Cell 1.4 ── Compile the model

cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled ✅")
print("\nOptimizer: Adam (lr=0.001)")
print("Loss:      sparse_categorical_crossentropy")
print("Metric:    accuracy")


In [ ]:
# ── Cell 1.4b ── Train the CNN
# This cell will take approximately 10-15 minutes with GPU enabled.
# Watch the accuracy and loss values update after each epoch.

# Callbacks — these automatically improve training:
callbacks = [
    # Stop training early if validation accuracy stops improving (saves time)
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True,
                  verbose=1),
    # Reduce learning rate if training gets stuck
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
]

print("Starting CNN training...")
print("Each epoch processes all", 17280 // 32, "training batches.\n")

history = cnn_model.fit(
    train_ds_final,
    epochs=30,                   # Maximum number of epochs (early stopping may end sooner)
    validation_data=val_ds_final,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Training complete!")


## 1.5 Evaluate the CNN

After training, we evaluate the model on the **test set** — images it has never seen before. This gives us an honest measure of how well the model generalizes.


In [ ]:
# ── Cell 1.5a ── Plot training history (accuracy and loss curves)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Training Accuracy', color='steelblue', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', color='darkorange', linewidth=2)
axes[0].set_title('Model Accuracy over Epochs', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0, 1])

# Loss
axes[1].plot(history.history['loss'], label='Training Loss', color='steelblue', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', color='darkorange', linewidth=2)
axes[1].set_title('Model Loss over Epochs', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('CNN Training History — EuroSAT Sentinel-2 Classification',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('cnn_training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n💾 Figure saved. Take a screenshot of this for your lab submission.")


In [ ]:
# ── Cell 1.5b ── Evaluate on the test set

print("Evaluating on held-out test set...")
test_loss, test_accuracy = cnn_model.evaluate(test_ds_final, verbose=0)
print(f"\n{'='*40}")
print(f"  Test Accuracy: {test_accuracy*100:.2f}%")
print(f"  Test Loss:     {test_loss:.4f}")
print(f"{'='*40}")


In [ ]:
# ── Cell 1.5c ── Confusion Matrix

# Get all predictions and true labels from the test set
y_true, y_pred = [], []
for images, labels in test_ds_final:
    preds = cnn_model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Plot confusion matrix
cm = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100  # Convert to %

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, linewidths=0.5)
ax.set_ylabel('True Label', fontweight='bold', fontsize=12)
ax.set_xlabel('Predicted Label', fontweight='bold', fontsize=12)
ax.set_title('Confusion Matrix — CNN on EuroSAT Test Set (% per true class)',
             fontsize=13, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('cnn_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n💾 Confusion matrix saved. Include this in your submission.")


In [ ]:
# ── Cell 1.5d ── Per-class accuracy report

print("Per-class accuracy report:")
print("="*60)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))


In [ ]:
# ── Cell 1.5e ── Visualize individual predictions

# Pick 20 random test images and show what the model predicted
test_images_list, test_labels_list = [], []
for images, labels in test_ds_final.take(5):
    test_images_list.extend(images.numpy())
    test_labels_list.extend(labels.numpy())

indices = random.sample(range(len(test_images_list)), 20)

fig, axes = plt.subplots(4, 5, figsize=(16, 13))
fig.suptitle("CNN Predictions on Test Images", fontsize=14, fontweight='bold')

for ax, idx in zip(axes.flatten(), indices):
    img = test_images_list[idx]
    true_label = CLASS_NAMES[test_labels_list[idx]]
    pred_probs = cnn_model.predict(img[np.newaxis, ...], verbose=0)[0]
    pred_label = CLASS_NAMES[np.argmax(pred_probs)]
    confidence = np.max(pred_probs) * 100

    ax.imshow(img)
    color = 'green' if pred_label == true_label else 'red'
    ax.set_title(f"True: {true_label}\nPred: {pred_label}\n({confidence:.1f}%)",
                 fontsize=8, color=color, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('cnn_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Green title = correct prediction | Red title = wrong prediction")


## 1.6 Your Turn — Experiment with the CNN

Now you'll make **one change** to the model and observe what happens. This is how real deep learning research works.


In [ ]:
# ── Cell 1.6 ── STUDENT EXPERIMENT: Change one hyperparameter
#
# INSTRUCTIONS:
# Choose ONE of the following changes and make it below:
#
# Option A: Change the learning rate
#   → Try: learning_rate=0.0001  or  learning_rate=0.01
#
# Option B: Add one more convolutional block
#   → Copy one Conv2D + MaxPooling block and add it before the Flatten layer
#
# Option C: Change dropout rate
#   → Try Dropout(0.1) instead of Dropout(0.25) — less regularization
#
# After making your change, run this cell and the training cell again.
# Compare your new accuracy with the original.

# ── MODIFY THIS FUNCTION ──────────────────────────────────────────────────

def build_cnn_experiment(input_shape=(64, 64, 3), num_classes=10):
    model = models.Sequential([

        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),   # ← Option C: change this value

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.4),

        # ← Option B: add a new block here

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

cnn_experiment = build_cnn_experiment()

cnn_experiment.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),  # ← Option A: change this
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_exp = cnn_experiment.fit(
    train_ds_final,
    epochs=15,
    validation_data=val_ds_final,
    callbacks=[EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True)],
    verbose=1
)

exp_loss, exp_acc = cnn_experiment.evaluate(test_ds_final, verbose=0)
print(f"\nOriginal model test accuracy: {test_accuracy*100:.2f}%")
print(f"Experiment model test accuracy: {exp_acc*100:.2f}%")
print(f"Change: {(exp_acc - test_accuracy)*100:+.2f}%")


### 📝 Reflection Questions — Part 1

Double-click this cell to edit and type your answers.


**Q1.1:** Look at your training accuracy curve. Is the training accuracy higher or lower than the validation accuracy? What does this tell you about whether the model is overfitting?

*Your answer here:*

---

**Q1.2:** Look at your confusion matrix. Which two land cover classes does the CNN confuse most often? Why do you think those two classes are hard to distinguish from satellite imagery?

*Your answer here:*

---

**Q1.3:** The CNN classifies an entire 64×64 image patch as one class. If a patch contained both a river and a forest, how would the model handle this? Is this a strength or a limitation?

*Your answer here:*

---

**Q1.4 (Experiment):** What change did you make in Cell 1.6? Did the test accuracy go up, down, or stay the same? Give one reason why you think it changed in that direction.

*Your answer here:*


---
# Part 2: U-Net for Land Cover Segmentation

---

## 2.1 From Classification to Segmentation

The CNN in Part 1 answered the question: **"What land cover class is this entire image patch?"**

But in real GIS applications, we often need a more detailed answer: **"What land cover class is *each individual pixel*?"**

This is called **semantic segmentation**. Instead of outputting one label per image, the model outputs a **label map** — an image of the same size as the input, where every pixel has its own predicted class.

### CNN vs U-Net

| | CNN | U-Net |
|---|---|---|
| **Task** | Image classification | Pixel-level segmentation |
| **Output** | One label per image | One label per pixel |
| **Example** | "This patch is Forest" | "These 1,240 pixels are Forest, these 830 are River..." |
| **Architecture** | Encoder only (shrinks the image) | Encoder + Decoder (shrinks then expands) |

### The U-Net Architecture

U-Net gets its name from its U-shaped architecture:

```
Input Image
    │
    ▼
[Encoder — Contracting Path]
  Conv → Pool → (feature maps get smaller, but deeper)
  Conv → Pool
  Conv → Pool
    │
    ▼
[Bottleneck] — Most compressed representation
    │
    ▼
[Decoder — Expanding Path]
  Upsample + Skip Connection → Conv
  Upsample + Skip Connection → Conv
  Upsample + Skip Connection → Conv
    │
    ▼
Output Segmentation Map (same size as input)
```

The key innovation is **skip connections** — direct links from the encoder to the decoder at each scale. These help the decoder recover fine spatial details (exactly *where* the boundary is) that were lost during pooling.


## 2.2 Prepare Data for Segmentation

For segmentation we need images **and** pixel-level label masks. We will use a subset of the **LandCover.ai** dataset — aerial/satellite imagery over Poland with hand-labeled masks for buildings, woodlands, water, and roads.

For this lab we use a smaller pre-processed version suitable for Colab.


In [ ]:
# ── Cell 2.2 ── Download and prepare segmentation dataset

# We use a preprocessed patch dataset derived from LandCover.ai
# compatible with Sentinel-2 style multispectral imagery
# Classes: 0=Background, 1=Building, 2=Woodland, 3=Water, 4=Road

!pip install -q gdown

import gdown, zipfile, pathlib

# Download preprocessed 256x256 segmentation patches
# (pre-tiled from LandCover.ai, RGB, with masks)
DATA_URL = "https://drive.google.com/uc?id=1lBy5T7lrMlRkB0MNDFgHpqR1oVzGhMkw"
DATA_PATH = pathlib.Path("landcover_patches")

if not DATA_PATH.exists():
    print("Downloading segmentation dataset...")
    gdown.download(DATA_URL, "landcover.zip", quiet=False)
    with zipfile.ZipFile("landcover.zip", 'r') as z:
        z.extractall(".")
    print("\n✅ Dataset downloaded and extracted!")
else:
    print("✅ Dataset already exists.")

# List what we have
image_paths = sorted(DATA_PATH.glob("images/*.png"))
mask_paths  = sorted(DATA_PATH.glob("masks/*.png"))
print(f"\nFound {len(image_paths)} image patches")
print(f"Found {len(mask_paths)} mask patches")


In [ ]:
# ── Cell 2.2b ── Fallback: generate synthetic segmentation data if download fails
# Run this cell ONLY if the download above failed.

import cv2

def generate_synthetic_segmentation_data(n_samples=400, img_size=128):
    """
    Creates synthetic image/mask pairs for demonstration.
    Each image has random colored regions; masks label each region.
    """
    X = np.zeros((n_samples, img_size, img_size, 3), dtype=np.float32)
    Y = np.zeros((n_samples, img_size, img_size, 1), dtype=np.int32)

    np.random.seed(42)
    for i in range(n_samples):
        # Random land cover simulation
        img = np.random.rand(img_size, img_size, 3).astype(np.float32)
        mask = np.zeros((img_size, img_size), dtype=np.int32)

        # Simulate 3-4 regions per image
        for cls in range(1, 5):
            cx, cy = np.random.randint(20, img_size-20, 2)
            r = np.random.randint(15, 40)
            color = [cls * 0.2, (4-cls) * 0.1, np.random.rand() * 0.3]
            cv2.circle(img, (cx, cy), r, color, -1)
            cv2.circle(mask, (cx, cy), r, cls, -1)

        X[i] = np.clip(img, 0, 1)
        Y[i, :, :, 0] = mask

    return X, Y

# Generate if needed
X_syn, Y_syn = generate_synthetic_segmentation_data(500, 128)
print("✅ Synthetic dataset ready (use this if download failed)")
print(f"   Image stack shape: {X_syn.shape}")
print(f"   Mask stack shape:  {Y_syn.shape}")
print("\nNote: Using synthetic data — results will be illustrative only.")


In [ ]:
# ── Cell 2.2c ── Load and preview the segmentation data

SEG_CLASSES = {
    0: ('Background', '#D3D3D3'),
    1: ('Building',   '#E74C3C'),
    2: ('Woodland',   '#27AE60'),
    3: ('Water',      '#2980B9'),
    4: ('Road',       '#F39C12')
}
NUM_SEG_CLASSES = len(SEG_CLASSES)
IMG_SIZE_SEG = 128

from PIL import Image as PILImage

def load_segmentation_data(image_paths, mask_paths, img_size=128, max_samples=400):
    X, Y = [], []
    for ip, mp in zip(image_paths[:max_samples], mask_paths[:max_samples]):
        img  = np.array(PILImage.open(ip).resize((img_size, img_size))) / 255.0
        mask = np.array(PILImage.open(mp).resize((img_size, img_size), PILImage.NEAREST))
        if img.ndim == 2: img = np.stack([img]*3, axis=-1)
        if img.shape[-1] == 4: img = img[:,:,:3]
        X.append(img.astype(np.float32))
        Y.append(mask[:,:,np.newaxis].astype(np.int32))
    return np.array(X), np.array(Y)

try:
    X_data, Y_data = load_segmentation_data(image_paths, mask_paths, IMG_SIZE_SEG)
    print(f"✅ Loaded real dataset: {X_data.shape[0]} samples")
except:
    X_data, Y_data = X_syn[:400], Y_syn[:400]
    IMG_SIZE_SEG = 128
    print("✅ Using synthetic dataset for demonstration")

# Train / validation / test split
X_train, X_temp, Y_train, Y_temp = train_test_split(X_data, Y_data, test_size=0.3, random_state=42)
X_val, X_test, Y_val, Y_test = train_test_split(X_temp, Y_temp, test_size=0.5, random_state=42)

print(f"\nTrain: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")


In [ ]:
# ── Cell 2.2d ── Visualize sample images and their masks

def colorize_mask(mask_2d):
    """Convert integer mask to RGB color image for display."""
    h, w = mask_2d.shape
    color_img = np.zeros((h, w, 3))
    colors_rgb = {
        0: [0.83, 0.83, 0.83],
        1: [0.91, 0.30, 0.24],
        2: [0.15, 0.68, 0.38],
        3: [0.16, 0.50, 0.73],
        4: [0.95, 0.61, 0.07]
    }
    for cls, rgb in colors_rgb.items():
        color_img[mask_2d == cls] = rgb
    return color_img

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle("Segmentation Dataset — Image, Mask, and Overlay", fontsize=13, fontweight='bold')

for row in range(3):
    idx = row * 40
    img  = X_train[idx]
    mask = Y_train[idx, :, :, 0]
    colored_mask = colorize_mask(mask)

    axes[row, 0].imshow(img)
    axes[row, 0].set_title("Satellite Image", fontweight='bold')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(colored_mask)
    axes[row, 1].set_title("Ground Truth Mask", fontweight='bold')
    axes[row, 1].axis('off')

    axes[row, 2].imshow(img)
    axes[row, 2].imshow(colored_mask, alpha=0.5)
    axes[row, 2].set_title("Overlay", fontweight='bold')
    axes[row, 2].axis('off')

# Legend
patches = [mpatches.Patch(color=c, label=SEG_CLASSES[i][0])
           for i, (n, c) in SEG_CLASSES.items()]
fig.legend(handles=patches, loc='lower center', ncol=5, fontsize=11,
           title='Land Cover Classes', title_fontsize=11, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig('segmentation_samples.png', dpi=150, bbox_inches='tight')
plt.show()


## 2.3 Build the U-Net Architecture

The U-Net is more complex than the CNN because it has both an encoder (contraction) and a decoder (expansion). We build it using small reusable blocks.


In [ ]:
# ── Cell 2.3 ── Define the U-Net model
# Architecture based on the original Ronneberger et al. (2015) U-Net paper
# Adapted from gicait DL-for-satellite-image-analysis (S5 — Land Cover Mapping)

def conv_block(x, filters, kernel_size=3, dropout_rate=0.1):
    """Two convolution layers with BatchNorm and optional dropout."""
    x = layers.Conv2D(filters, kernel_size, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters, kernel_size, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    if dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)
    return x

def encoder_block(x, filters):
    """One encoder step: conv_block + MaxPooling."""
    skip = conv_block(x, filters)      # skip connection saved here
    pool = layers.MaxPooling2D((2, 2))(skip)
    return skip, pool

def decoder_block(x, skip, filters):
    """One decoder step: Upsample + concatenate skip + conv_block."""
    x = layers.Conv2DTranspose(filters, (2, 2), strides=2, padding='same')(x)
    x = layers.Concatenate()([x, skip])  # ← skip connection from encoder
    x = conv_block(x, filters)
    return x

def build_unet(input_shape=(128, 128, 3), num_classes=5):
    """Full U-Net architecture."""
    inputs = layers.Input(shape=input_shape)

    # ── Encoder (contracting path) ────────────────────────────────
    s1, p1 = encoder_block(inputs, 32)   # 128→64,  32 filters
    s2, p2 = encoder_block(p1,     64)   # 64→32,   64 filters
    s3, p3 = encoder_block(p2,    128)   # 32→16,  128 filters

    # ── Bottleneck ────────────────────────────────────────────────
    b = conv_block(p3, 256, dropout_rate=0.2)   # 16×16×256

    # ── Decoder (expanding path) ──────────────────────────────────
    d1 = decoder_block(b,  s3, 128)    # 16→32
    d2 = decoder_block(d1, s2, 64)     # 32→64
    d3 = decoder_block(d2, s1, 32)     # 64→128

    # ── Output layer ──────────────────────────────────────────────
    # One filter per class, softmax gives per-pixel class probabilities
    outputs = layers.Conv2D(num_classes, (1, 1), activation='softmax')(d3)

    model = models.Model(inputs, outputs, name="U-Net")
    return model

unet_model = build_unet(input_shape=(IMG_SIZE_SEG, IMG_SIZE_SEG, 3), num_classes=NUM_SEG_CLASSES)
unet_model.summary()


## 2.4 Train the U-Net


In [ ]:
# ── Cell 2.4 ── Compile and train U-Net

unet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Training U-Net... (approximately 10-20 minutes with GPU)")

Y_train_sq = Y_train[:,:,:,0]  # Remove last dim for sparse_categorical_crossentropy
Y_val_sq   = Y_val[:,:,:,0]

unet_history = unet_model.fit(
    X_train, Y_train_sq,
    epochs=30,
    batch_size=16,
    validation_data=(X_val, Y_val_sq),
    callbacks=[
        EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
    ],
    verbose=1
)

print("\n✅ U-Net training complete!")


In [ ]:
# ── Cell 2.4b ── Plot U-Net training history

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(unet_history.history['accuracy'],     label='Train',      color='steelblue', lw=2)
axes[0].plot(unet_history.history['val_accuracy'], label='Validation', color='darkorange', lw=2)
axes[0].set_title('U-Net Pixel Accuracy over Epochs', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_ylim([0,1])

axes[1].plot(unet_history.history['loss'],     label='Train',      color='steelblue', lw=2)
axes[1].plot(unet_history.history['val_loss'], label='Validation', color='darkorange', lw=2)
axes[1].set_title('U-Net Loss over Epochs', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('U-Net Training History — Land Cover Segmentation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('unet_training_history.png', dpi=150, bbox_inches='tight')
plt.show()


## 2.5 Evaluate and Visualize Predictions

Now let's see what the U-Net actually produces — pixel-level land cover maps.


In [ ]:
# ── Cell 2.5a ── Generate predictions on test set

Y_test_sq = Y_test[:,:,:,0]
test_loss, test_acc = unet_model.evaluate(X_test, Y_test_sq, verbose=0)
print(f"U-Net Test Pixel Accuracy: {test_acc*100:.2f}%")
print(f"U-Net Test Loss:           {test_loss:.4f}")

# Get predicted masks
Y_pred_prob = unet_model.predict(X_test, verbose=0)
Y_pred = np.argmax(Y_pred_prob, axis=-1)   # Shape: (N, H, W)


In [ ]:
# ── Cell 2.5b ── Visualize predictions vs ground truth

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
fig.suptitle("U-Net Predictions — Satellite Image / Ground Truth / Prediction / Overlay",
             fontsize=13, fontweight='bold')

col_titles = ['Input Image', 'Ground Truth', 'U-Net Prediction', 'Prediction Overlay']
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontweight='bold', fontsize=11)

sample_indices = random.sample(range(len(X_test)), 4)

for row, idx in enumerate(sample_indices):
    img        = X_test[idx]
    true_mask  = Y_test_sq[idx]
    pred_mask  = Y_pred[idx]

    axes[row, 0].imshow(img);                        axes[row, 0].axis('off')
    axes[row, 1].imshow(colorize_mask(true_mask));   axes[row, 1].axis('off')
    axes[row, 2].imshow(colorize_mask(pred_mask));   axes[row, 2].axis('off')
    axes[row, 3].imshow(img)
    axes[row, 3].imshow(colorize_mask(pred_mask), alpha=0.55)
    axes[row, 3].axis('off')

# Legend
patches = [mpatches.Patch(color=c, label=SEG_CLASSES[i][0])
           for i, (n, c) in SEG_CLASSES.items()]
fig.legend(handles=patches, loc='lower center', ncol=5, fontsize=11,
           bbox_to_anchor=(0.5, -0.01))
plt.tight_layout()
plt.savefig('unet_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n💾 Figure saved. Include this in your submission.")


In [ ]:
# ── Cell 2.5c ── Per-class IoU (Intersection over Union)
# IoU is a better metric for segmentation than simple accuracy
# IoU = (True Positives) / (True Positives + False Positives + False Negatives)

def compute_iou(y_true_flat, y_pred_flat, num_classes):
    iou_scores = []
    for cls in range(num_classes):
        intersection = np.sum((y_true_flat == cls) & (y_pred_flat == cls))
        union        = np.sum((y_true_flat == cls) | (y_pred_flat == cls))
        iou = intersection / union if union > 0 else float('nan')
        iou_scores.append(iou)
    return iou_scores

y_true_flat = Y_test_sq.flatten()
y_pred_flat = Y_pred.flatten()
iou_scores  = compute_iou(y_true_flat, y_pred_flat, NUM_SEG_CLASSES)

print("Per-class IoU scores:")
print("-" * 35)
for cls_idx, (name, color) in SEG_CLASSES.items():
    iou = iou_scores[cls_idx]
    bar = "█" * int(iou * 30) if not np.isnan(iou) else "N/A"
    print(f"  {name:<18} {iou:.3f}  {bar}")

mean_iou = np.nanmean(iou_scores)
print("-" * 35)
print(f"  Mean IoU (mIoU):   {mean_iou:.3f}")
print()
print("Note: IoU of 1.0 = perfect overlap. IoU < 0.5 = poor segmentation.")


### 📝 Reflection Questions — Part 2

Double-click this cell to edit and write your answers.


**Q2.1:** What is the key structural difference between a CNN (Part 1) and a U-Net (Part 2)? In your own words, explain what a "skip connection" does and why it matters for producing accurate segmentation maps.

*Your answer here:*

---

**Q2.2:** Look at the prediction maps from Cell 2.5b. Find one example where the U-Net made a clear mistake. Describe what the model predicted versus what the ground truth shows. Why do you think it made that error?

---

**Q2.3:** The U-Net gives you a per-pixel land cover map. Describe one real GIS workflow where this kind of output would be directly useful — be specific about what you would do with the segmentation map after producing it.

*Your answer here:*

---

**Q2.4:** Compare the pixel accuracy (from Cell 2.5a) with the per-class IoU scores (from Cell 2.5c). Which class had the lowest IoU? Is a high overall accuracy always a reliable indicator of good segmentation performance? Explain why or why not.

*Your answer here:*


---
# Part 3: Summary and Submission

---

## 3.1 What We Did in This Lab

| Step | What you did | Tool |
|---|---|---|
| Part 0 | Set up Colab, learned Python basics, explored Sentinel-2 images as arrays | NumPy, Matplotlib |
| Part 1 | Built, trained, and evaluated a CNN for land use classification on EuroSAT | TensorFlow/Keras |
| Part 1.6 | Modified a hyperparameter and compared results | Experimentation |
| Part 2 | Built and trained a U-Net for pixel-level land cover segmentation | TensorFlow/Keras |
| Part 2 | Evaluated with IoU, visualized predicted masks vs ground truth | NumPy, Matplotlib |

---

## 3.2 Final Reflection Questions

Answer these in the cell below.


**Q3.1:** In your own words, explain to a non-technical audience (e.g. a city planner) what deep learning for satellite imagery does and why it is useful. Keep it to 3–4 sentences.

*Your answer here:*

---

**Q3.2:** Both models in this lab were trained on European imagery. If you wanted to apply them to classify land cover in southern Illinois, what problems might you encounter? How would you address them?

*Your answer here:*

---

**Q3.3:** Deep learning models require large amounts of labeled training data. For a custom project (e.g. mapping wetlands in a specific region), you might only have 200 labeled image patches. What strategies could you use to still build a working model with limited data?

*Your answer here:*

---

**Q3.4:** What is one limitation of using RGB (3-band) imagery compared to full Sentinel-2 multispectral data (13 bands) for land cover classification? Which land cover class do you think would benefit most from additional spectral bands, and why?

*Your answer here:*


---
## 3.3 Submission Instructions

Before downloading your notebook, run the cell below to create a summary of your results.


In [ ]:
# ── Cell 3.3 ── Generate a results summary for your submission

print("="*55)
print("        LAB 8 RESULTS SUMMARY")
print("="*55)
print(f"\n  CNN — EuroSAT Land Use Classification")
print(f"    Test Accuracy:  {test_accuracy*100:.2f}%")
print(f"    Num Classes:    {NUM_CLASSES}")
print(f"    Architecture:   3-block CNN + Dense head")

print(f"\n  U-Net — Land Cover Segmentation")
print(f"    Test Pixel Acc: {test_acc*100:.2f}%")
print(f"    Mean IoU:       {mean_iou:.3f}")
print(f"    Num Classes:    {NUM_SEG_CLASSES}")
print(f"    Architecture:   3-level U-Net with skip connections")

print(f"\n  Experiment (Part 1.6)")
print(f"    Original Acc:   {test_accuracy*100:.2f}%")
try:
    print(f"    Experiment Acc: {exp_acc*100:.2f}%")
    print(f"    Change:         {(exp_acc-test_accuracy)*100:+.2f}%")
except:
    print(f"    (Run Cell 1.6 to populate)")
print("="*55)
print("\n📋 To submit this lab:")
print("   1. Make sure all cells have been run (Cell → Run All)")
print("   2. File → Download → Download .ipynb")
print("   3. Rename: Lastname_Lab8_DeepLearning.ipynb")
print("   4. Upload to D2L Assignments")
print("\nScreenshots to include in your submission:")
print("   ✓ eurosat_samples.png")
print("   ✓ cnn_training_history.png")
print("   ✓ cnn_confusion_matrix.png")
print("   ✓ unet_training_history.png")
print("   ✓ unet_predictions.png")


---

## 📚 References and Further Reading

- Helber, P., et al. (2019). EuroSAT: A novel dataset and deep learning benchmark for land use and land cover classification. *IEEE Journal of Selected Topics in Applied Earth Observations and Remote Sensing*, 12(7), 2217–2226.

- Ronneberger, O., Fischer, P., & Brox, T. (2015). U-Net: Convolutional networks for biomedical image segmentation. *MICCAI 2015*, Lecture Notes in Computer Science.

- GICAIT. (2022). *Deep Learning for Satellite Image Analysis*. GitHub. https://github.com/gicait/DL-for-satellite-image-analysis

- Copernicus / ESA. (2024). *Sentinel-2 Mission*. https://sentinel.esa.int/web/sentinel/missions/sentinel-2

---
*Lab 8 — Advanced Spatial Analysis | Southern Illinois University | GeoFEW Lab*
